In [1]:

import tensorflow as tf
import sys
sys.path.append("..")
from functions import BunDLeNet, BunDLeTrainer

model = BunDLeNet(3)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
trainer = BunDLeTrainer(model, optimizer)
all_losses=[]
epochs = 5

from data_reader import DataReader
from processed_data_storage import ProcessedDataStroage
import numpy as np
from preprocess import Preprocessor

data_reader = DataReader()
processed_data_storage = ProcessedDataStroage()

for sub in range(1, 6):
    subject = sub+100
    subject = str(subject)
    raw, channels, events = data_reader.read_data(
        subject=str(subject),
    )

    preprocessor = Preprocessor(raw)
    del raw

    preprocessor.run_pipeline(use_filter=True, use_ica=False, use_autoreject=False)

    epochs_train, epochs_test= preprocessor.split_train_test()

    X_train, X_test = preprocessor.scale(epochs_train.get_data(), epochs_test.get_data())
    X_train = X_train.astype(np.float32)
    X_test = X_test.astype(np.float32)

    labels_train, category_per_epoch= preprocessor.create_labels_for_epochs(epochs_train)    # Form (1638, 250)
    labels_test, category_per_epoch= preprocessor.create_labels_for_epochs(epochs_test)     # Form (410, 250)
    labels_train = labels_train.astype(np.float32)
    labels_test = labels_test.astype(np.float32)

    processed_data_storage.save_subject(subject, {"X_train":X_train, "X_test":X_test, "labels_train":labels_train, "labels_test":labels_test})

I0000 00:00:1794691677.704463   28479 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794691678.083674   28479 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794691679.865084   28479 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


None


W0000 00:00:1794691681.732605   28479 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1794691681.735625   28479 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1794691681.866285   28479 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9217 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:03:00.0, compute capability: 12.0a


Arbeitsverzeichnis: /home/bened/bachelor-thesis/notebooks/BunDLe-Net/notebooks
Datenverzeichnis: /home/bened/bachelor-thesis/data/raw/ds005363-download
Lese BrainVision-Aufnahme ein: /home/bened/bachelor-thesis/data/raw/ds005363-download/sub-101/ses-01/eeg/sub-101_ses-01_task-objrecog_eeg.vhdr
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 pas

In [2]:
X_train = []
X_test = []
labels_train = []
labels_test = []
subjects_range = range(1,6)

for sub in subjects_range:
    subject = sub+100
    values = processed_data_storage.load_subject(subject)    
    X_train.append(values[0])
    X_test.append(values[1])
    labels_train.append(values[2])
    labels_test.append(values[3])

X_train = np.concatenate(X_train, axis=0)
X_test = np.concatenate(X_test, axis=0)
labels_train = np.concatenate(labels_train, axis=0)
labels_test = np.concatenate(labels_test, axis=0)

print(X_train.shape)
print(X_test.shape)
print(labels_train.shape)
print(labels_test.shape)

(8190, 64, 120)
(2050, 64, 120)
(8190, 120)
(2050, 120)
